# MedSAM2: segment a 3D CT by prompting a single slice

SAM 2 propagates a prompt across video frames with memory attention. A CT volume has
the same structure — adjacent slices differ only slightly — so one box on one slice can
be tracked through the whole stack.

**Runtime → Change runtime type → T4 GPU** before running anything.

- MedSAM2: https://github.com/bowang-lab/MedSAM2
- Weights: https://huggingface.co/wanglab/MedSAM2 (research and education only)
- Paper: https://arxiv.org/abs/2504.03600

## 1 · Environment

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun."
)
print(torch.cuda.get_device_name(0))

In [ ]:
%%capture
!pip install -q git+https://github.com/rekalantar/medsam2-3d-ct.git
!pip install -q SimpleITK imageio huggingface_hub

In [ ]:
%%capture
!git clone -q https://github.com/bowang-lab/MedSAM2.git /content/MedSAM2
%cd /content/MedSAM2
!pip install -q -e ".[dev]"
!bash download.sh

In [ ]:
import os

CKPT = "/content/MedSAM2/checkpoints/MedSAM2_latest.pt"
assert os.path.exists(CKPT), "download.sh did not produce MedSAM2_latest.pt"
print(f"checkpoint {os.path.getsize(CKPT) / 1e6:.0f} MB")

## 2 · Data

The MedSAM2 demo dataset ships per-case NIfTI volumes with ground-truth masks. Fetch
one case — a few MB — rather than calling `load_dataset`, which would pull tens of
gigabytes.

Having the label means we can place the prompt automatically and score the result.

In [ ]:
from huggingface_hub import hf_hub_download

DATASET = "wanglab/CT_DeepLesion-MedSAM2"
CASE = "000009_03_01_036-048"          # ~4 MB; see the listing cell below for others

image_path = hf_hub_download(
    DATASET, filename=f"images/{CASE}_0000.nii.gz", repo_type="dataset")
label_path = hf_hub_download(
    DATASET, filename=f"labels/{CASE}.nii.gz", repo_type="dataset")

print(image_path)
print(label_path)

In [ ]:
# Optional: other cases, smallest first.
from huggingface_hub import HfApi

info = HfApi().repo_info(DATASET, repo_type="dataset", files_metadata=True)
imgs = sorted((s.size, s.rfilename) for s in info.siblings
              if s.rfilename.startswith("images/"))
for size, name in imgs[:10]:
    print(f"{size/1e6:6.1f} MB  {name.split('/')[1].replace('_0000.nii.gz', '')}")

## 3 · Load and window

The step people skip. CT spans several thousand Hounsfield units; the network takes
8-bit. Min-maxing the full range collapses soft tissue into a handful of grey levels.

In [ ]:
import numpy as np
from medsam2_ct import load_volume, window_hu

volume_hu, spacing = load_volume(image_path)     # (z, y, x), spacing in mm
truth, _ = load_volume(label_path)
truth = truth > 0

volume = window_hu(volume_hu, width=400, level=40)

print(f"volume  {volume.shape}  spacing {tuple(round(s, 2) for s in spacing)}")
print(f"lesion  {truth.sum():,} voxels across {truth.any(axis=(1,2)).sum()} slices")

## 4 · Place the prompt

Pick the slice where the lesion is largest, and take its bounding box with a small
margin — a stand-in for a clinician drawing one.

This uses the ground-truth label, which is how promptable models are normally
evaluated: it simulates a perfect user prompt and isolates the propagation from the
question of whether a human drew a good box.

In [ ]:
MARGIN = 5

areas = truth.sum(axis=(1, 2))
KEY_SLICE = int(areas.argmax())

ys, xs = np.where(truth[KEY_SLICE])
BOX = [int(xs.min()) - MARGIN, int(ys.min()) - MARGIN,
       int(xs.max()) + MARGIN, int(ys.max()) + MARGIN]

print(f"key slice {KEY_SLICE} of {len(volume)}  ({areas[KEY_SLICE]:,} voxels)")
print(f"box {BOX}")

In [ ]:
import matplotlib.patches as patches
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(volume[KEY_SLICE], cmap="gray")
ax.contour(truth[KEY_SLICE], levels=[0.5], colors="#3FC1C9", linewidths=1.5)
ax.add_patch(patches.Rectangle(
    (BOX[0], BOX[1]), BOX[2] - BOX[0], BOX[3] - BOX[1],
    edgecolor="#FF6B5B", facecolor="none", linewidth=2))
ax.set_title(f"prompt on slice {KEY_SLICE}  (truth in cyan)")
ax.axis("off")

## 5 · One box, whole volume

`segment_volume` propagates forward and backward. The key slice sits mid-lesion, so
forward-only would capture roughly half.

In [ ]:
from medsam2_ct import build_predictor, largest_component, segment_volume

predictor = build_predictor(config="configs/sam2.1_hiera_t512.yaml", checkpoint=CKPT)

masks = segment_volume(predictor, volume, BOX, KEY_SLICE)
print(f"raw            {masks.sum():,} voxels, {masks.any(axis=(1,2)).sum()} slices")

masks = largest_component(masks)
print(f"largest comp   {masks.sum():,} voxels, {masks.any(axis=(1,2)).sum()} slices")
print(f"ground truth   {truth.sum():,} voxels, {truth.any(axis=(1,2)).sum()} slices")

## 6 · Score it

In [ ]:
def dice(a, b):
    a, b = a.astype(bool), b.astype(bool)
    d = a.sum() + b.sum()
    return 1.0 if d == 0 else 2 * (a & b).sum() / d

print(f"Dice  {dice(truth, masks):.3f}")

# Dice on the prompted slice alone, versus the volume as a whole:
# the gap is the cost of propagation.
print(f"Dice on the key slice  {dice(truth[KEY_SLICE], masks[KEY_SLICE]):.3f}")

## 7 · The payoff figure

Watch it rather than trusting the number. Failure modes are obvious to the eye and
invisible in a mean. Keep the GIF under 8 MB or Medium rejects it.

In [ ]:
from medsam2_ct import save_gif

z = (masks | truth).any(axis=(1, 2)).nonzero()[0]
lo, hi = max(0, z.min() - 4), min(len(volume), z.max() + 5)

save_gif(volume[lo:hi], masks[lo:hi], "propagation.gif", fps=6)
print(f"{os.path.getsize('propagation.gif') / 1e6:.1f} MB, {hi - lo} frames")

In [ ]:
from IPython.display import Image

Image("propagation.gif")

## 8 · Check the claims

Three things the write-up asserts. If one doesn't hold on your data, change the
write-up, not the result.

**Claim 1 — windowing changes the result.**

In [ ]:
naive = ((volume_hu - volume_hu.min()) /
         (volume_hu.max() - volume_hu.min()) * 255).astype(np.uint8)

masks_naive = largest_component(segment_volume(predictor, naive, BOX, KEY_SLICE))

print(f"windowed  Dice {dice(truth, masks):.3f}  ({masks.sum():,} voxels)")
print(f"naive     Dice {dice(truth, masks_naive):.3f}  ({masks_naive.sum():,} voxels)")

**Claim 2 — backward propagation matters.** Forward-only should lose roughly half.

In [ ]:
from medsam2_ct import init_state

state, _h, _w = init_state(predictor, volume)
predictor.add_new_points_or_box(
    inference_state=state, frame_idx=KEY_SLICE, obj_id=1,
    box=np.asarray(BOX, dtype=np.float32),
)

fwd = np.zeros_like(masks)
for idx, _ids, logits in predictor.propagate_in_video(state):
    fwd[idx] = (logits[0] > 0).cpu().numpy().squeeze()

print(f"bidirectional  {masks.sum():,} voxels  Dice {dice(truth, masks):.3f}")
print(f"forward only   {fwd.sum():,} voxels  Dice {dice(truth, fwd):.3f}"
      f"  ({fwd.sum() / masks.sum():.0%} of bidirectional)")

**Claim 3 — accuracy decays with distance from the prompt.**

Per-slice Dice against distance from the prompted slice. If it stays flat to both
extremes, that paragraph of the write-up is wrong and should say so.

In [ ]:
z = truth.any(axis=(1, 2)).nonzero()[0]
per_slice = [dice(truth[i], masks[i]) for i in z]

plt.figure(figsize=(8, 3.5))
plt.plot(z - KEY_SLICE, per_slice, "o-", color="#3FC1C9", linewidth=2)
plt.axvline(0, color="#FF6B5B", linestyle="--", label="prompted slice")
plt.xlabel("slices from prompt"); plt.ylabel("Dice")
plt.ylim(0, 1.05); plt.legend(); plt.tight_layout()

---

Project: https://github.com/rekalantar/medsam2-3d-ct